# Qwen 2.5 1.5B GRPO RL Training — Modal A100 40GB
**Reinforcement Learning via GRPO on `abhinav0231/reasoning-mixed-3600`**

Single A100 40GB setup on Modal with Unsloth, Flash Attention 2, BFloat16 native support, HF step weight checkpointing, Lossless 16-bit model merging, and **Custom Rollout Text Logger (`/root/grpo_rollouts_log.txt`)**.


## Cell 1 — Install Dependencies (Modal %uv optimized)

In [ ]:
# ==============================================================================
# Cell 1 — Dependency Installation (Modal %uv Fast Package Manager)
# ==============================================================================
# Install latest Unsloth optimized for GRPO RL training from GitHub
%uv pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q

# Install deep learning libraries & TRL (Transformer Reinforcement Learning):
# - wandb: Experiment tracking & reward metric visualization
# - flash-attn & liger-kernel: Fast CUDA kernels for LLM policy updates
# - trl: GRPOTrainer and GRPOConfig implementation
%uv pip install wandb flash-attn liger-kernel trl datasets huggingface_hub -q

## Cell 2 — Hardware Probe & Hardware Check

In [ ]:
# ==============================================================================
# Cell 2 — Hardware Probe & Unsloth Verification
# ==============================================================================
# IMPORTANT: 'import unsloth' must be imported first to apply fast CUDA patches
import unsloth
print(f"Unsloth Version : {unsloth.__version__}")

import torch, trl
print(f"PyTorch Version : {torch.__version__}")
print(f"TRL Version     : {trl.__version__}")
print(f"CUDA Version    : {torch.version.cuda}")

# Probe GPU compute capability and VRAM capacity
p = torch.cuda.get_device_properties(0)
print(f"\nGPU Device     : {p.name}")
print(f"VRAM Capacity  : {p.total_memory / 1e9:.1f} GB")
print(f"Compute        : cc={p.major}.{p.minor}")
print(f"BFloat16       : {'Supported' if p.major >= 8 else 'NOT supported'}")
print(f"FlashAttention2: {'Supported' if p.major >= 8 else 'NOT supported'}")

# Assert hardware compatibility for Modal A100
assert torch.cuda.is_available(), "No GPU detected!"
assert p.major >= 8, f"Requires Ampere+ GPU (A100/H100/RTX3090+). Got cc={p.major}.{p.minor}"
print("\n✅ Hardware check passed for Modal A100")

## Cell 3 — Configuration & Hyperparameters

In [ ]:
# ==============================================================================
# Cell 3 — Configuration & Hyperparameters
# ==============================================================================
import os, torch

HF_USERNAME = "abhinav0231"

# Base Model & LoRA Parameters
# Loads the merged 16-bit SFT Warmup checkpoint created in Notebook 02
MODEL_NAME     = f"{HF_USERNAME}/Qwen2.5-1.5B-reasoning-warmup-merged"

# ------------------------------------------------------------------------------
# Prompt Len (1024) + Completion Len (1536) = 2560 tokens.
# 3072 is set to provide a 512-token safety buffer for ChatML template tags,
# system prompt overhead, and GPU memory tile alignment (power-of-two blocks),
# preventing CUDA OOMs and rollout truncation mid-sentence.
# ------------------------------------------------------------------------------
MAX_SEQ_LENGTH = 3072
LORA_RANK      = 64       # Increased LoRA rank r=64 for high RL policy capacity
LORA_ALPHA     = 64       # LoRA scaling factor alpha=64

# Target Dataset Repo on Hugging Face Hub (3,600 multi-domain RL prompts)
DATASET_REPO   = f"{HF_USERNAME}/reasoning-mixed-3600"

# ------------------------------------------------------------------------------
# WHY MAX_STEPS = 600?
# Dataset size = 3,600 prompts.
# Effective Batch Size = BATCH_SIZE (4) * GRAD_ACCUM (4) = 16 samples per step.
# 1 full epoch over dataset = 3,600 / 16 = 225 steps.
# 600 max steps corresponds to ~2.67 full epochs of RL training,
# providing sufficient policy rollouts for reward convergence without over-fitting.
# ------------------------------------------------------------------------------
MAX_STEPS       = 600
LEARNING_RATE   = 5e-6     # Conservative RL learning rate for stable policy optimization
WARMUP_RATIO    = 0.05     # 5% linear warmup
NUM_GENERATIONS = 8        # G = 8 rollout completions per prompt for GRPO advantage calculation
BATCH_SIZE      = 4        # Per-device batch size
GRAD_ACCUM      = 4        # Gradient accumulation steps (Effective Batch = 16)
SEED            = 42
TEMPERATURE     = 0.9      # Sampling temperature for diverse rollout exploration
MAX_PROMPT_LEN  = 1024     # Maximum prompt sequence length
MAX_COMP_LEN    = 1536     # Maximum rollout completion generation length

# Output Repositories on Hugging Face Hub
HF_ADAPTER_REPO = f"{HF_USERNAME}/Qwen2.5-1.5B-GRPO-adapter"
HF_MERGED_REPO  = f"{HF_USERNAME}/Lily-1.5B"                # Final standalone GRPO model
CHECKPOINT_REPO = f"{HF_USERNAME}/Qwen2.5-1.5B-GRPO-checkpoints"
SAVE_STEPS      = 50

# Local Filesystem Paths on Modal Container
OUTPUT_DIR       = "/root/grpo_output"
MERGED_DIR       = "/root/grpo_merged"
ROLLOUT_LOG_FILE = "/root/grpo_rollouts_log.txt"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(MERGED_DIR, exist_ok=True)

# W&B Experiment Tracking Config
WANDB_PROJECT  = "superqwen"
WANDB_RUN_NAME = "qwen2.5-1.5b-grpo-a100"

DTYPE = torch.bfloat16
print(f"Base Model      : {MODEL_NAME}")
print(f"Dataset         : {DATASET_REPO}")
print(f"Adapter Repo    : {HF_ADAPTER_REPO}")
print(f"Merged Repo     : {HF_MERGED_REPO}")
print(f"Checkpoints Repo: {CHECKPOINT_REPO}")
print(f"Rollouts Log    : {ROLLOUT_LOG_FILE}")
print(f"Batch Config    : Batch={BATCH_SIZE}, GradAccum={GRAD_ACCUM} => EffBatch={BATCH_SIZE*GRAD_ACCUM}")

## Cell 4 — Authentication (Hugging Face & W&B)

In [ ]:
# ==============================================================================
# Robust Authentication (Hugging Face & Weights & Biases)
# ==============================================================================
import os
from huggingface_hub import login, HfFolder

# Hugging Face Authentication
HF_TOKEN = os.environ.get("HF_TOKEN", "")
if not HF_TOKEN or HF_TOKEN == "YOUR_HF_TOKEN_HERE":
    cached_token = HfFolder.get_token()
    if cached_token:
        HF_TOKEN = cached_token

if HF_TOKEN and HF_TOKEN != "YOUR_HF_TOKEN_HERE":
    try:
        login(token=HF_TOKEN)
        print("✅ Authenticated with Hugging Face")
    except Exception as e:
        print(f"⚠️ Hugging Face authentication note: {e}")
else:
    print("ℹ️ HF_TOKEN not provided. Proceeding (public datasets/models remain accessible).")

# WandB Authentication
try:
    import wandb
    WANDB_TOKEN = os.environ.get("WANDB_API_KEY", "")
    if WANDB_TOKEN and WANDB_TOKEN != "YOUR_WANDB_KEY_HERE":
        wandb.login(key=WANDB_TOKEN, relogin=True)
        os.environ["WANDB_API_KEY"] = WANDB_TOKEN
        print("✅ Authenticated with Weights & Biases")
    else:
        print("ℹ️ WANDB_API_KEY not found. WandB tracking will operate in offline/disabled mode.")
        os.environ["WANDB_DISABLED"] = "true"
except ImportError:
    print("ℹ️ WandB module not installed. Operating without WandB tracking.")
    os.environ["WANDB_DISABLED"] = "true"


## Cell 5 — Load Unsloth Model & Setup GRPO PEFT LoRA

In [ ]:
# ==============================================================================
# Cell 5 — Unsloth Fast Model Loading & GRPO LoRA Initialization
# ==============================================================================
from unsloth import FastLanguageModel

print(f"Loading base model: {MODEL_NAME} ...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = DTYPE,
    load_in_4bit   = True,
)

tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = "right"

# Configure PEFT LoRA adapters (rank r=64, alpha=64) on all 7 target linear layers
model = FastLanguageModel.get_peft_model(
    model,
    r = LORA_RANK,
    lora_alpha = LORA_ALPHA,
    target_modules = ["q_proj","k_proj","v_proj","o_proj",
                      "gate_proj","up_proj","down_proj"],
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = SEED,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable/1e6:.1f}M / {total/1e6:.0f}M ({100*trainable/total:.1f}%")

## Cell 6 — Load & Format GRPO Dataset

In [ ]:
# ==============================================================================
# Cell 6 — Load Dataset & Format for GRPOTrainer
# ==============================================================================
from datasets import load_dataset

SYSTEM_PROMPT = (
    "You are a precise, helpful assistant. "
    "Always reason step by step inside <think></think> tags, "
    "then write your final answer inside <answer></answer> tags."
)

# Format dataset dicts into GRPOTrainer prompt list structure
def format_for_grpo(example):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": example["prompt"]},
    ]
    return {
        "prompt":      messages,
        "answer":      example["answer"],
        "answer_type": example["answer_type"],
        "source":      example["source"],
    }

print(f"Loading GRPO dataset from HF: {DATASET_REPO} ...")
dataset = (
    load_dataset(DATASET_REPO, split="train")
    .shuffle(seed=SEED)
    .map(format_for_grpo, batched=False)
)
print(f"✅ Dataset loaded: {len(dataset):,} samples")
print(f"Sample prompt format:\n{dataset[0]['prompt']}")

## Cell 7 — Reward Helpers & Domain Reward Functions

In [ ]:
# ==============================================================================
# Cell 7 — Multi-Domain GRPO Reward Functions
# ==============================================================================
import re

# Text extraction & normalization helpers
def _to_text(comp):
    if isinstance(comp, str):
        return comp
    if isinstance(comp, list):
        return "\n".join(m.get("content", "") if isinstance(m, dict) else str(m) for m in comp)
    return str(comp)

def _extract_think(text):
    m = re.search(r"<think>(.*?)</think>", text, re.DOTALL)
    return m.group(1).strip() if m else ""

def _extract_answer(text):
    m = re.search(r"<answer>(.*?)</answer>", text, re.DOTALL)
    return m.group(1).strip() if m else text.strip()

def _extract_number(text):
    nums = re.findall(r"-?\d+(?:,\d{3})*(?:\.\d+)?", text)
    return nums[-1].replace(",", "") if nums else None

def _normalize_math(text):
    text = str(text).strip()
    text = re.sub(r"\\boxed\{(.*?)\}", r"\1", text)
    text = re.sub(r"\$+", "", text)
    text = re.sub(r"\\text\{(.*?)\}", r"\1", text)
    return text.replace(",", "").replace(" ", "").lower().strip()

def _math_equal(pred, gold):
    pred_n, gold_n = _extract_number(pred), _extract_number(gold)
    if pred_n and gold_n:
        try:
            return abs(float(pred_n) - float(gold_n)) < 1e-3
        except ValueError:
            pass
    if _normalize_math(pred) == _normalize_math(gold):
        return True
    def _frac(s):
        parts = s.strip().split("/")
        if len(parts) == 2:
            try: return float(parts[0]) / float(parts[1])
            except: pass
        try: return float(s)
        except: return None
    pf, gf = _frac(_normalize_math(pred)), _frac(_normalize_math(gold))
    if pf is not None and gf is not None:
        return abs(pf - gf) < 1e-3
    return False

# ------------------------------------------------------------------------------
# Reward Function 1: Format Compliance (Applies to all completions)
# Checks presence of <think>...</think> and <answer>...</answer> tags.
# Adds bonus for thinking body >= 50 words.
# ------------------------------------------------------------------------------
def format_reward(completions, **kwargs):
    rewards = []
    for comp in completions:
        comp = _to_text(comp)
        score       = 0.0
        think_body  = _extract_think(comp)
        answer_body = _extract_answer(comp)
        has_think   = "<think>" in comp and "</think>" in comp
        has_answer  = "<answer>" in comp and "</answer>" in comp

        if has_think:
            score += 0.2
            if len(think_body.split()) >= 50:
                score += 0.15
        else:
            score -= 0.15

        if has_answer:
            score += 0.15
            if answer_body:
                score += 0.10
        else:
            score -= 0.15

        rewards.append(float(round(score, 4)))
    return rewards

# ------------------------------------------------------------------------------
# Reward Function 2: Correctness (Routed dynamically by answer_type)
# Evaluates prediction accuracy against ground-truth for:
# - 'numeric': GSM8K math numbers
# - 'exact': MATH-Hard, OpenR1-Math, ARC-Challenge multiple choice letters (A-D)
# - 'bool': StrategyQA Yes/No boolean answers
# - 'code': HumanEval/MBPP Python function syntax (def, return, 4-space indent)
# ------------------------------------------------------------------------------
def correctness_reward(completions, answer=None, answer_type=None, **kwargs):
    if answer is None or answer_type is None:
        return [0.0] * len(completions)

    rewards = []
    for comp, ans, atype in zip(completions, answer, answer_type):
        comp = _to_text(comp)
        pred = _extract_answer(comp)

        if atype == "numeric":
            correct = _math_equal(pred, str(ans))
            rewards.append(1.0 if correct else -0.5)
        elif atype == "exact":
            gold_upper = str(ans).strip().upper()
            pred_upper = pred.strip().upper()
            if re.fullmatch(r"[A-D]", gold_upper):
                letter = re.search(r"\b([ABCD])\b", pred_upper)
                predicted = letter.group(1) if letter else None
                rewards.append(1.0 if predicted == gold_upper else -0.5)
            else:
                rewards.append(1.0 if _math_equal(pred, str(ans)) else -0.5)
        elif atype == "bool":
            pred_lower = pred.lower().strip()
            gold_lower = str(ans).lower().strip()
            yes_match  = "yes" in pred_lower
            no_match   = "no" in pred_lower
            if gold_lower == "yes":
                rewards.append(1.0 if yes_match and not no_match else -0.5)
            else:
                rewards.append(1.0 if no_match and not yes_match else -0.5)
        elif atype == "code":
            has_def    = bool(re.search(r"^\s*def\s+\w+", pred, re.MULTILINE))
            has_return = "return" in pred
            has_indent = bool(re.search(r"^\s{4}", pred, re.MULTILINE))
            score = sum([has_def, has_return, has_indent]) / 3.0
            rewards.append(round(score * 0.8 - 0.2, 3))
        else:
            rewards.append(0.0)
    return rewards

# ------------------------------------------------------------------------------
# Reward Function 3: Instruction Adherence (Applies to Alpaca prompts)
# ------------------------------------------------------------------------------
def instruction_reward(completions, prompt=None, answer_type=None, source=None, **kwargs):
    if prompt is None or answer_type is None:
        return [0.0] * len(completions)
    rewards = []
    _source = source if source is not None else [None] * len(completions)
    for comp, p, atype, src in zip(completions, prompt, answer_type, _source):
        if src != "alpaca" or p is None:
            rewards.append(0.0)
            continue
        comp = _to_text(comp)
        answer_text = _extract_answer(comp)
        prompt_lower = p.lower() if isinstance(p, str) else "".join(m.get("content", "") for m in p).lower()
        ok, n = 0, 0
        wc_m = re.search(r"(\d+)\s*words?", prompt_lower)
        if wc_m:
            n += 1
            target = int(wc_m.group(1))
            if abs(len(answer_text.split()) - target) <= max(5, int(target * 0.12)):
                ok += 1
        if n > 0:
            ratio = ok / n
            rewards.append(round(ratio * 1.0 - (1 - ratio) * 0.5, 3))
        else:
            wc = len(answer_text.split())
            rewards.append(0.3 if 15 <= wc <= 600 else -0.1)
    return rewards

# ------------------------------------------------------------------------------
# Reward Function 4: Length Guard Penalty
# Penalizes overly short outputs (<15 words) or runaway loops (>2500 words).
# ------------------------------------------------------------------------------
def length_reward(completions, **kwargs):
    rewards = []
    for comp in completions:
        comp = _to_text(comp)
        wc = len(comp.split())
        if wc < 15:       rewards.append(-0.5)
        elif wc > 2500:   rewards.append(-0.25)
        else:             rewards.append(0.15)
    return rewards

print("✅ Reward functions configured")

## Cell 8 — Rollout Text File Logger & Checkpoint Callbacks

In [ ]:
# ==============================================================================
# Cell 8 — Custom Callbacks (Rollout Logger & HF Step Checkpoint Pusher)
# ==============================================================================
from transformers import TrainerCallback
from huggingface_hub import upload_folder, HfApi
import datetime

# ------------------------------------------------------------------------------
# 1. Rollout Text File Logger Callback
# Writes detailed step logs (loss, KL divergence, rewards, completion length)
# to readable file on Modal filesystem: /root/grpo_rollouts_log.txt
# ------------------------------------------------------------------------------
class RolloutTextLoggerCallback(TrainerCallback):
    def __init__(self, log_filepath: str):
        self.log_filepath = log_filepath
        with open(self.log_filepath, "w", encoding="utf-8") as f:
            f.write(f"==================================================\n")
            f.write(f"  GRPO TRAINING ROLLOUT LOG — {datetime.datetime.now().isoformat()}\n")
            f.write(f"==================================================\n\n")
        print(f"✅ RolloutTextLoggerCallback initialized -> {self.log_filepath}")

    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs:
            return
        step = state.global_step
        loss = logs.get("loss", None)
        kl = logs.get("kl", logs.get("objective/kl", None))
        reward = logs.get("reward", logs.get("reward/total", None))
        comp_len = logs.get("completion_length", None)

        log_entry = f"--- Step {step:04d} Metrics ---\n"
        if loss is not None:     log_entry += f"  Loss              : {loss:.4f}\n"
        if kl is not None:       log_entry += f"  KL Divergence     : {kl:.4f}\n"
        if reward is not None:   log_entry += f"  Mean Reward       : {reward:.4f}\n"
        if comp_len is not None: log_entry += f"  Completion Length : {comp_len:.1f} tokens\n"
        log_entry += "-" * 50 + "\n\n"

        with open(self.log_filepath, "a", encoding="utf-8") as f:
            f.write(log_entry)

# ------------------------------------------------------------------------------
# 2. Intermediate Checkpoint Push Callback
# Pushes local checkpoint adapters to HF Hub after every save step (SAVE_STEPS=50).
# ------------------------------------------------------------------------------
class CheckpointPushCallback(TrainerCallback):
    def __init__(self, repo_id: str, token: str):
        self.repo_id = repo_id
        self.token   = token
        if repo_id:
            try:
                HfApi().create_repo(repo_id, token=token, exist_ok=True, private=True)
                print(f"Checkpoint repo ready: {repo_id}")
            except Exception as e:
                print(f"WARNING: could not create checkpoint repo: {e}")

    def on_save(self, args, state, control, **kwargs):
        step = state.global_step
        ckpt_dir = os.path.join(args.output_dir, f"checkpoint-{step}")
        if not os.path.exists(ckpt_dir):
            return
        print(f"\n>> Step {step}: uploading checkpoint to HF Hub -> {self.repo_id} ...")
        try:
            upload_folder(
                folder_path     = ckpt_dir,
                repo_id         = self.repo_id,
                token           = self.token,
                path_in_repo    = f"checkpoint-{step}",
                commit_message  = f"GRPO checkpoint step {step}",
                ignore_patterns = ["*.lock"],
            )
            print(f">> Step {step} checkpoint uploaded.")
        except Exception as e:
            print(f"WARNING: HF push failed at step {step}: {e}")

_callbacks = [
    RolloutTextLoggerCallback(log_filepath=ROLLOUT_LOG_FILE),
    CheckpointPushCallback(repo_id=CHECKPOINT_REPO, token=HF_TOKEN)
]
print("✅ Callbacks configured successfully")

## Cell 9 — Configure GRPOTrainer & Execute RL Training

In [ ]:
# ==============================================================================
# Cell 9 — GRPOTrainer Setup & RL Policy Loop Execution
# ==============================================================================
from trl import GRPOConfig, GRPOTrainer

# 1. Initialize W&B Experiment Tracking
wandb.init(
    project  = WANDB_PROJECT,
    name     = WANDB_RUN_NAME,
    settings = wandb.Settings(init_timeout=300),
    config   = {
        "model": MODEL_NAME, "dataset": DATASET_REPO,
        "lora_rank": LORA_RANK, "max_steps": MAX_STEPS,
        "lr": LEARNING_RATE, "num_generations": NUM_GENERATIONS,
        "batch_size": BATCH_SIZE, "grad_accum": GRAD_ACCUM,
        "temperature": TEMPERATURE, "dtype": str(DTYPE),
    },
)

# 2. Configure GRPO Algorithm Parameters
training_args = GRPOConfig(
    output_dir                  = OUTPUT_DIR,
    use_vllm                    = False,
    num_generations             = NUM_GENERATIONS,   # G = 8 rollouts
    max_prompt_length           = MAX_PROMPT_LEN,    # 1024
    max_completion_length       = MAX_COMP_LEN,      # 1536
    temperature                 = TEMPERATURE,
    top_p                       = 0.95,
    learning_rate               = LEARNING_RATE,
    lr_scheduler_type           = "cosine",
    warmup_ratio                = WARMUP_RATIO,
    max_steps                   = MAX_STEPS,         # 600 max steps (~2.67 epochs)
    per_device_train_batch_size = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,
    loss_type                   = "dapo",            # Dual-clip Advantage Policy Optimization
    epsilon                     = 0.2,               # PPO clipping epsilon
    epsilon_high                = 0.28,              # Upper clipping threshold
    beta                        = 0.001,             # KL divergence coefficient
    mask_truncated_completions  = True,
    fp16                        = False,
    bf16                        = True,
    optim                       = "adamw_torch_fused",
    dataloader_num_workers      = 0,
    logging_steps               = 5,
    save_steps                  = SAVE_STEPS,
    save_total_limit            = 3,
    report_to                   = "wandb",
    run_name                    = WANDB_RUN_NAME,
    seed                        = SEED,
)

# 3. Instantiate GRPOTrainer with the 4 reward functions
trainer = GRPOTrainer(
    model            = model,
    reward_funcs     = [format_reward, correctness_reward, instruction_reward, length_reward],
    args             = training_args,
    train_dataset    = dataset,
    processing_class = tokenizer,
    callbacks        = _callbacks,
)

print(f"\n{'='*60}")
print(f"  GRPO Training (Modal A100 40GB) — {MODEL_NAME}")
print(f"  Eff. Batch = {BATCH_SIZE} x {GRAD_ACCUM} = {BATCH_SIZE*GRAD_ACCUM} | Generations = {NUM_GENERATIONS}")
print(f"  Steps: {MAX_STEPS} | LR: {LEARNING_RATE} | Warmup: {WARMUP_RATIO}")
print(f"{'='*60}\n")

# 4. Launch RL Policy Training Loop
stats = trainer.train()
print(f"\n✅ GRPO Training completed!")
wandb.finish()

## Cell 10 — Save & Push Trained GRPO LoRA Adapter

In [ ]:
# ==============================================================================
# Cell 10 — Save & Push GRPO LoRA Adapter to HF Hub
# ==============================================================================
print(f"Saving trained GRPO LoRA adapter locally to {OUTPUT_DIR}/lora_adapter ...")
model.save_pretrained(f"{OUTPUT_DIR}/lora_adapter")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/lora_adapter")

print(f"Pushing LoRA adapter to Hugging Face Hub: {HF_ADAPTER_REPO} ...")
model.push_to_hub(HF_ADAPTER_REPO, token=HF_TOKEN)
tokenizer.push_to_hub(HF_ADAPTER_REPO, token=HF_TOKEN)
print(f"✅ LoRA adapter pushed to https://huggingface.co/{HF_ADAPTER_REPO}")

## Cell 11 — Lossless 16-bit Model Merging & Push (`Lily-1.5B`)

In [ ]:
# ==============================================================================
# Cell 11 — Lossless 16-bit Model Merging & Hub Deployment (Lily-1.5B)
# ==============================================================================
from unsloth import FastLanguageModel

# 1. Reload base warmup model + trained GRPO adapter in full BFloat16 precision
print("Loading trained model + GRPO adapter in full precision BFloat16 for lossless merge ...")
model_m, tok_m = FastLanguageModel.from_pretrained(
    model_name     = OUTPUT_DIR,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = torch.bfloat16,  # Full precision BF16 merge
    load_in_4bit   = False,           # Disable quantization artifacts
)

# 2. Merge GRPO LoRA adapter parameters into base model weights
print(f"Merging LoRA weights locally into 16-bit model -> {MERGED_DIR} ...")
model_m.save_pretrained_merged(MERGED_DIR, tok_m, save_method="merged_16bit")
print(f"✅ Merged model saved locally to {MERGED_DIR}")

# 3. Push final 16-bit standalone model 'Lily-1.5B' to Hugging Face Hub Hub
print(f"Pushing final merged 16-bit model to Hugging Face Hub: {HF_MERGED_REPO} ...")
model_m.push_to_hub_merged(HF_MERGED_REPO, tok_m, save_method="merged_16bit", token=HF_TOKEN)
print(f"\n🎉 Successfully merged and pushed final 16-bit model Lily-1.5B to:")
print(f"   https://huggingface.co/{HF_MERGED_REPO}")